# Day 083 — Exercise 1: Task Representation and Parsing the Plan

**What you'll build:** the `Task` dataclass, a list-aware JSON parser (`safe_parse_list`), a planning prompt, and `parse_plan` — the function that turns raw LLM output into a structured task list.

**Why it matters:** decomposition starts with a representation. Every task needs an id (so others can depend on it), a title, a description (what to do), and a dependency list. And like every LLM parser this section, `parse_plan` must tolerate garbage output — missing fields, prose, invalid JSON — without crashing.

In [ ]:
import json

_PLAN_JSON = json.dumps([
    {'id': 't1', 'title': 'Gather facts',
     'description': 'Collect the relevant information.', 'depends_on': []},
    {'id': 't2', 'title': 'Draft outline',
     'description': 'Organize the facts into an outline.', 'depends_on': ['t1']},
    {'id': 't3', 'title': 'Write summary',
     'description': 'Write the final summary.', 'depends_on': ['t2']},
])

def _mock_planner(plan_json=None, task_result='Task done.'):
    """Return an llm_fn: the plan JSON on planning calls, task_result on execution calls."""
    plan = plan_json if plan_json is not None else _PLAN_JSON
    def _fn(messages):
        system = messages[0]['content'] if messages else ''
        if 'json array' in system.lower() or 'planning' in system.lower():
            return plan
        return task_result
    return _fn

def _mock_executor(task):
    return 'Result: ' + task.title
# Task dataclass needs dataclasses stdlib; parse_plan uses safe_parse_list below.


## Task

1. `safe_parse_list(text)` — like `safe_parse_json` but finds `[` to `]`; returns a `list` or `None`; never raises.
2. `Task` dataclass — fields: `id: str`, `title: str`, `description: str`, `depends_on: list = []`, `status: str = 'pending'`, `result: str = ''`.
3. `build_plan_prompt(goal, context=None)` — a `system` message instructing the LLM to return **only** a JSON array of task objects; a `user` message with the goal.
4. `parse_plan(text) -> list[Task]` — `safe_parse_list(text) or []`; build a `Task` for each dict item, filling missing fields with safe defaults. **Never raises.**

## Your Implementation

In [ ]:
from dataclasses import dataclass, field
import json

def safe_parse_list(text):
    """Slice first '[' to last ']' and parse. Returns list|None. Never raises."""
    raise NotImplementedError

@dataclass
class Task:
    """One step in a plan: id, title, description, depends_on, status, result."""
    id: str = ''             # TODO: add all six fields with correct defaults
    title: str = ''
    description: str = ''

def build_plan_prompt(goal, context=None):
    """Build a prompt asking the LLM to decompose goal into a JSON task list."""
    raise NotImplementedError

def parse_plan(text):
    """Extract list[Task] from LLM output. Never raises; missing fields -> defaults."""
    raise NotImplementedError


In [ ]:
import json
from dataclasses import dataclass, field

# ── helpers reused from Day 79 ───────────────────────────────────────────────
def safe_parse_json(text):
    """Slice first '{' to last '}' and parse. Returns dict|None (Day 79)."""
    start, end = text.find("{"), text.rfind("}")
    if start == -1 or end == -1 or end < start:
        return None
    try:
        data = json.loads(text[start:end + 1])
    except (json.JSONDecodeError, ValueError):
        return None
    return data if isinstance(data, dict) else None


def safe_parse_list(text):
    """Slice first '[' to last ']' and parse. Returns list|None. Never raises."""
    start, end = text.find("["), text.rfind("]")
    if start == -1 or end == -1 or end < start:
        return None
    try:
        data = json.loads(text[start:end + 1])
    except (json.JSONDecodeError, ValueError):
        return None
    return data if isinstance(data, list) else None


def call_llm(messages, llm_fn=None):
    """Call the chat model, or the injected llm_fn(messages) -> str (Day 79)."""
    if llm_fn is not None:
        return llm_fn(messages)
    import ollama
    resp = ollama.chat(model="llama3.2", messages=messages)
    return resp["message"]["content"]

# ── the Task dataclass ────────────────────────────────────────────────────────
@dataclass
class Task:
    """One step in a plan.

    Attributes:
        id:          short snake_case identifier (e.g. 't1', 'write_outline').
        title:       brief human-readable label (5 words max).
        description: one sentence describing what to do.
        depends_on:  ids of tasks that must complete before this one.
        status:      'pending' | 'done' | 'failed'.
        result:      the output of executing this task.
    """
    id: str
    title: str
    description: str
    depends_on: list = field(default_factory=list)
    status: str = "pending"
    result: str = ""

# ── planning: ask the LLM to decompose a goal ─────────────────────────────────
def build_plan_prompt(goal, context=None):
    """Build a prompt that asks the LLM to break a goal into a JSON task list."""
    system = "\n".join([
        "You are a planning assistant. Break the goal into an ordered list of tasks.",
        "",
        "Return ONLY a JSON array. Each item must have these exact keys:",
        '  "id": short snake_case identifier (t1, t2, ...)',
        '  "title": brief label (5 words max)',
        '  "description": one sentence - what to do',
        '  "depends_on": list of task ids that must finish before this one ([] if none)',
        "",
        "Return ONLY the JSON array. No prose, no markdown fences.",
    ])
    user_parts = ["Goal: " + str(goal)]
    if context:
        user_parts.append("Context: " + str(context))
    return [{"role": "system", "content": system},
            {"role": "user", "content": "\n".join(user_parts)}]


def parse_plan(text):
    """Extract a task list from LLM output. Returns list[Task]; never raises.

    Tolerates markdown fences, prose before/after, missing fields, and invalid
    JSON. Invalid or missing fields are filled with safe defaults so any
    parseable item becomes a valid Task.
    """
    items = safe_parse_list(text) or []
    tasks = []
    for i, item in enumerate(items):
        if not isinstance(item, dict):
            continue
        tasks.append(Task(
            id=str(item.get("id", "t" + str(i + 1))),
            title=str(item.get("title", "Task " + str(i + 1))),
            description=str(item.get("description", "")),
            depends_on=[str(d) for d in item.get("depends_on", [])
                        if isinstance(d, str)],
        ))
    return tasks


## Automated checks

In [ ]:

score, total = 0, 5
try:
    t = Task(id='t1', title='Step', description='Do it.')
    assert t.status == 'pending' and t.result == '' and t.depends_on == []
    score += 1; print("✅ Task has correct fields and defaults")

    assert safe_parse_list('[1, 2, 3]') == [1, 2, 3]
    assert safe_parse_list('{"a":1}') is None     # dict, not list
    assert safe_parse_list('no json here') is None
    score += 1; print("✅ safe_parse_list parses lists, returns None for non-lists")

    msgs = build_plan_prompt('write a report')
    assert msgs[0]['role'] == 'system' and 'json array' in msgs[0]['content'].lower()
    assert msgs[1]['content'].startswith('Goal:')
    score += 1; print("✅ build_plan_prompt asks for a JSON array")

    tasks = parse_plan(_PLAN_JSON)
    assert len(tasks) == 3 and tasks[0].id == 't1' and tasks[1].depends_on == ['t1']
    score += 1; print("✅ parse_plan extracts a valid task list")

    assert parse_plan('no json at all') == []
    t_bad = parse_plan('[{"id":"x"}]')
    assert len(t_bad) == 1 and t_bad[0].title != ''   # missing fields get defaults
    score += 1; print("✅ parse_plan returns [] on garbage; missing fields get defaults")

except Exception as e:
    print(f"❌ {e}")

print(f"\n{score}/{total} checks passed")
if score == total:
    print("\U0001f389 Exercise complete!")


## Solution

<details><summary>Reveal</summary>

```python
import json
from dataclasses import dataclass, field

# ── helpers reused from Day 79 ───────────────────────────────────────────────
def safe_parse_json(text):
    """Slice first '{' to last '}' and parse. Returns dict|None (Day 79)."""
    start, end = text.find("{"), text.rfind("}")
    if start == -1 or end == -1 or end < start:
        return None
    try:
        data = json.loads(text[start:end + 1])
    except (json.JSONDecodeError, ValueError):
        return None
    return data if isinstance(data, dict) else None


def safe_parse_list(text):
    """Slice first '[' to last ']' and parse. Returns list|None. Never raises."""
    start, end = text.find("["), text.rfind("]")
    if start == -1 or end == -1 or end < start:
        return None
    try:
        data = json.loads(text[start:end + 1])
    except (json.JSONDecodeError, ValueError):
        return None
    return data if isinstance(data, list) else None


def call_llm(messages, llm_fn=None):
    """Call the chat model, or the injected llm_fn(messages) -> str (Day 79)."""
    if llm_fn is not None:
        return llm_fn(messages)
    import ollama
    resp = ollama.chat(model="llama3.2", messages=messages)
    return resp["message"]["content"]

# ── the Task dataclass ────────────────────────────────────────────────────────
@dataclass
class Task:
    """One step in a plan.

    Attributes:
        id:          short snake_case identifier (e.g. 't1', 'write_outline').
        title:       brief human-readable label (5 words max).
        description: one sentence describing what to do.
        depends_on:  ids of tasks that must complete before this one.
        status:      'pending' | 'done' | 'failed'.
        result:      the output of executing this task.
    """
    id: str
    title: str
    description: str
    depends_on: list = field(default_factory=list)
    status: str = "pending"
    result: str = ""

# ── planning: ask the LLM to decompose a goal ─────────────────────────────────
def build_plan_prompt(goal, context=None):
    """Build a prompt that asks the LLM to break a goal into a JSON task list."""
    system = "\n".join([
        "You are a planning assistant. Break the goal into an ordered list of tasks.",
        "",
        "Return ONLY a JSON array. Each item must have these exact keys:",
        '  "id": short snake_case identifier (t1, t2, ...)',
        '  "title": brief label (5 words max)',
        '  "description": one sentence - what to do',
        '  "depends_on": list of task ids that must finish before this one ([] if none)',
        "",
        "Return ONLY the JSON array. No prose, no markdown fences.",
    ])
    user_parts = ["Goal: " + str(goal)]
    if context:
        user_parts.append("Context: " + str(context))
    return [{"role": "system", "content": system},
            {"role": "user", "content": "\n".join(user_parts)}]


def parse_plan(text):
    """Extract a task list from LLM output. Returns list[Task]; never raises.

    Tolerates markdown fences, prose before/after, missing fields, and invalid
    JSON. Invalid or missing fields are filled with safe defaults so any
    parseable item becomes a valid Task.
    """
    items = safe_parse_list(text) or []
    tasks = []
    for i, item in enumerate(items):
        if not isinstance(item, dict):
            continue
        tasks.append(Task(
            id=str(item.get("id", "t" + str(i + 1))),
            title=str(item.get("title", "Task " + str(i + 1))),
            description=str(item.get("description", "")),
            depends_on=[str(d) for d in item.get("depends_on", [])
                        if isinstance(d, str)],
        ))
    return tasks
```

**Why fill defaults for missing fields?** The model might return `{"id": "t1"}` without a title. Raising on a partial item would drop a task that could be valid enough to execute. Defaults keep the plan alive; the human can spot a blank title and fix it.

</details>